In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
excel_path = "/home/ubuntu/giodir/digitalPathology/data/aiFlopp/Trento AIFLOPP.xlsx"
sheet_name = "CONFRONTO REFERTI DEFINITIVO"

In [3]:
# open excel sheet in pandas

df = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0, 1])

/home/ubuntu/giodir/digitalPathology/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [4]:
df.head()

Unnamed: 0_level_0 REFERTI REGGIO                 \
          ID PATIENT    CODICE CASO REPERE/VETRINO   
0               TN01          Tn001              1   
1               TN01          Tn001              2   
2               TN01          Tn001              3   
3               TN01          Tn001              4   
4               TN01          Tn001              5   

                                                  \
  NUMERO FRUSTOLI PER REPERE/VETRINO     LETTORE   
0                                1.0  RAGAZZI M.   
1                                1.0  RAGAZZI M.   
2                                2.0  RAGAZZI M.   
3                                2.0  RAGAZZI M.   
4                                1.0  RAGAZZI M.   

                                                                                                                                                                                                                                                \
  DIAGNOSI (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)   
0                                                  4                                                                                                                                                                                             
1                                                  4                                                                                                                                                                                             
2                                                  4                                                                                                                                                                                             
3                                                  4                                                                                                                                                                                             
4                                                  4                                                                                                                                                                                             

                                                                                                                                                                                                                                                            \
  DIAGNOSI DOPO IMMUNO (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)   
0                                                NaN                                                                                                                                                                                                         
1                                                NaN                                                                                                                                                                                                         
2                                                NaN                                                                                                                                                                                                         
3                                                NaN                                                                                                                                                                                                         
4                                      

In [5]:
df.columns.to_list()

[('Unnamed: 0_level_0', 'ID PATIENT'),
 ('REFERTI REGGIO', 'CODICE CASO'),
 ('REFERTI REGGIO', 'REPERE/VETRINO'),
 ('REFERTI REGGIO', 'NUMERO FRUSTOLI PER REPERE/VETRINO'),
 ('REFERTI REGGIO', 'LETTORE'),
 ('REFERTI REGGIO',
  'DIAGNOSI (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)'),
 ('REFERTI REGGIO',
  'DIAGNOSI DOPO IMMUNO (0=no tumore; 1=ASAP/ATYP che richiede approfondimento; 2=PIN di alto grado; 3=AK intraduttale; 4=positivo per adenocarcinoma acinare; 5=positvo per adenocarcinoma duttale; N.A.=frustolo senza evidenza di ghiandole prostatiche)'),
 ('REFERTI REGGIO', 'GLEASON Principale'),
 ('REFERTI REGGIO', 'GLEASON Secondario'),
 ('REFERTI REGGIO', '% PATTERN 3'),
 ('REFERTI REGGIO', '% PATTERN 4'),
 ('REFERTI REGGIO', '% PATTERN 5'),
 ('REFERTI REGGIO', 'LUNGHEZZA CORES BIOPTICI (mm)'),
 ('REFERT

In [6]:
columns_to_keep = [
    ('REFERTI REGGIO', 'CODICE CASO'),
    ('REFERTI REGGIO', 'REPERE/VETRINO'),
    ('REFERTI REGGIO', 'GG ISUP PER SINGOLO REPERE/VETRINO'),
    ('REFERTI TRENTO', 'GG ISUP PER SINGOLO CORE'),
]

columns_new_names = {
    ('REFERTI REGGIO', 'CODICE CASO'): 'patient_id',
    ('REFERTI REGGIO', 'REPERE/VETRINO'): 'bersaglio',
    ('REFERTI REGGIO', 'GG ISUP PER SINGOLO REPERE/VETRINO'): 'GG_reggio',
    ('REFERTI TRENTO', 'GG ISUP PER SINGOLO CORE'): 'GG_trento',
}


normalized_cols = [tuple(x.strip() if isinstance(x, str) else x for x in col)
                   if isinstance(col, tuple) else col
                   for col in df.columns]
df.columns = pd.MultiIndex.from_tuples(normalized_cols)  # only if they are tuples

filtered_df = df[columns_to_keep]
filtered_df.columns = [columns_new_names.get(col, col) for col in filtered_df.columns]

In [7]:
filtered_df.head()

,patient_id,bersaglio,GG_reggio,GG_trento
0,Tn001,1,5.0,5.0
1,Tn001,2,5.0,5.0
2,Tn001,3,5.0,5.0
3,Tn001,4,5.0,5.0
4,Tn001,5,4.0,5.0


In [8]:
# Assign -1 values to case where GG is missing (meaning there is no tumor)
filtered_df['GG_reggio'] = filtered_df['GG_reggio'].fillna(-1)
filtered_df['GG_trento'] = filtered_df['GG_trento'].fillna(-1)

In [9]:
filtered_df["difference"] = np.abs(filtered_df['GG_reggio'] - filtered_df['GG_trento'])

In [10]:
filtered_df["difference"].value_counts()

difference
0.0    344
1.0     88
2.0     23
3.0      8
5.0      3
Name: count, dtype: int64

In [11]:
len(filtered_df)

466

In [12]:
# Parse labels to get the bag_id

parsed_case_code = filtered_df["patient_id"].apply(lambda x: str(int(str(x)[2:])))
filtered_df['tn_bag_id'] = "TN_" + parsed_case_code + "_" + filtered_df["bersaglio"].astype(str)

In [13]:
# Associate to each row the correct bag id looking at the folder

features_dir = Path("/home/ubuntu/giodir/digitalPathology/data/features/uni_features_TR")

def find_bag_id(tn_bag_id):
    for file in features_dir.glob(f"{tn_bag_id}_*.npz"):
        return file.stem  # return the filename without extension
    return None  # if no file is found

filtered_df['bag_id'] = filtered_df['tn_bag_id'].apply(find_bag_id)

print("not matched")
filtered_df[filtered_df["bag_id"].isnull()]

not matched


,patient_id,bersaglio,GG_reggio,GG_trento,difference,tn_bag_id,bag_id


In [14]:
filtered_df

,patient_id,bersaglio,GG_reggio,GG_trento,difference,tn_bag_id,bag_id
0,Tn001,1,5.0,5.0,0.0,TN_1_1,TN_1_1_113600
1,Tn001,2,5.0,5.0,0.0,TN_1_2,TN_1_2_113717
2,Tn001,3,5.0,5.0,0.0,TN_1_3,TN_1_3_113829
3,Tn001,4,5.0,5.0,0.0,TN_1_4,TN_1_4_114005
4,Tn001,5,4.0,5.0,1.0,TN_1_5,TN_1_5_114134
...,...,...,...,...,...,...,...
461,Tn040,8,-1.0,-1.0,0.0,TN_40_8,TN_40_8_174826
462,Tn040,9,-1.0,-1.0,0.0,TN_40_9,TN_40_9_174954
463,Tn040,10,-1.0,-1.0,0.0,TN_40_10,TN_40_10_175130
464,Tn040,11,-1.0,-1.0,0.0,TN_40_11,TN_40_11_175256


In [ ]:
# Filter to keep only the analyzed cases

features_dir = Path("/home/ubuntu/giodir/digitalPathology/data/features/uni_features_TR")

available_bags = {path.stem for path in features_dir.glob("*.npz")}
print("Analyzed bags:", len(available_bags))

avail_df = filtered_df[filtered_df['bag_id'].isin(available_bags)]

print(f"Total cases: {len(filtered_df)}, Available cases: {len(avail_df)},")

Analyzed bags: 466
Total cases: 466, Available cases: 466,


## LABEL TYPE

In [16]:
## binary like 0 vs 1+
avail_df["binary_difference"] = (avail_df["difference"] > 0).astype(int)

# keep only the important differences (set 0 where the difference is 0, 1 if it is 2 or more and None if it is 1)
avail_df["important_difference"] = avail_df["difference"].apply(lambda x: 0 if x == 0 else (1 if x >= 2 else None))

In [17]:
avail_df["binary_difference"].value_counts()

binary_difference
0    344
1    122
Name: count, dtype: int64

In [18]:
# Save in csv the three files like (bag_id, label_col)

basedir = Path("/home/ubuntu/giodir/digitalPathology/data/labels/tn_discordance_labels")
basedir.mkdir(exist_ok=True)


# binary diff
avail_df[["bag_id", "binary_difference"]].rename(columns={"binary_difference": "label"}).to_csv(
    basedir / "binary_diff_labels.csv", index=False)

# binary important diff
avail_df[["bag_id", "important_difference"]].rename(columns={"important_difference": "label"}).to_csv(
    basedir / "binary_important_diff_labels.csv", index=False)

# original diff
avail_df[["bag_id", "difference"]].rename(columns={"difference": "label"}).to_csv(
    basedir / "difference_labels.csv", index=False)
